
# 🧠 CV Demo — «Как нейросеть видит мир» (ResNet18 Feature Maps)

Минимальный интерактив:
1. Загружаем предобученную `ResNet18`.
2. Кладём картинку (URL или загрузка файла).
3. Визуализируем **карты признаков** первых слоёв.
4. Ползунками выбираем слой/канал и сразу видим, что происходит.

> Цель: почувствовать, что на ранних слоях сети появляются **контуры и текстуры**, а дальше — более сложные паттерны.


In [ ]:

# ⬇️ Зависимости (перезапустите среду после установки при первом запуске в Colab, если потребуется)
!pip -q install torch torchvision pillow ipywidgets matplotlib


In [ ]:

# ▶️ Импорты и базовая настройка
import torch
import torchvision
from torchvision import transforms
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
from io import BytesIO
import requests
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Предобученная ResNet18 (Torch >=2.0 API совместим; используем weights)
weights = torchvision.models.ResNet18_Weights.DEFAULT
model = torchvision.models.resnet18(weights=weights).to(device).eval()

# Подготовка препроцессинга из weights (даёт корректную нормализацию)
preprocess = weights.transforms()

# Соберём фичи с помощью forward hooks
feature_maps = {}
def get_hook(name):
    def hook(m, i, o):
        feature_maps[name] = o.detach().cpu()
    return hook

# Повесим hook-и на первые несколько блоков
hooks = []
layers_to_hook = {
    'conv1': model.conv1,
    'layer1_0_conv1': model.layer1[0].conv1,
    'layer1_0_conv2': model.layer1[0].conv2,
    'layer2_0_conv1': model.layer2[0].conv1,
}
for name, layer in layers_to_hook.items():
    hooks.append(layer.register_forward_hook(get_hook(name)))

def load_image_from_url(url):
    resp = requests.get(url, timeout=10)
    return Image.open(BytesIO(resp.content)).convert('RGB')

def load_image_from_upload(upload_widget):
    key = list(upload_widget.value.keys())[0]
    return Image.open(BytesIO(upload_widget.value[key]['content'])).convert('RGB')

def run_inference(img):
    with torch.no_grad():
        x = preprocess(img).unsqueeze(0).to(device)
        _ = model(x)

def visualize_feature_map(layer_name, channel_idx, vmin=None, vmax=None):
    fmap = feature_maps[layer_name][0]  # (C, H, W)
    c = fmap[channel_idx].numpy()
    plt.figure(figsize=(4,4))
    plt.imshow(c, cmap='gray', vmin=vmin, vmax=vmax)
    plt.axis('off')
    plt.title(f"{layer_name} · channel {channel_idx}")
    plt.show()

# UI
url_text = widgets.Text(
    value="https://images.unsplash.com/photo-1518791841217-8f162f1e1131",
    description='Image URL:',
    layout=widgets.Layout(width='80%')
)
upload = widgets.FileUpload(accept='image/*', multiple=False, description='Загрузить файл')
load_btn = widgets.Button(description='Загрузить и прогнать', button_style='primary')
layer_dropdown = widgets.Dropdown(options=list(layers_to_hook.keys()), description='Слой:')
channel_slider = widgets.IntSlider(value=0, min=0, max=63, step=1, description='Канал:')
auto_scale = widgets.Checkbox(value=True, description='Авто-контраст')
vmin_box = widgets.FloatText(value=None, description='vmin')
vmax_box = widgets.FloatText(value=None, description='vmax')

output = widgets.Output()

def on_load_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            if upload.value:
                img = load_image_from_upload(upload)
            else:
                img = load_image_from_url(url_text.value)
            display(img.resize((256,256)))
            run_inference(img)
            # Обновим верхнюю границу слайдера каналов по размеру первого выбранного слоя
            layer = layer_dropdown.value
            max_ch = feature_maps[layer].shape[1] - 1
            channel_slider.max = int(max_ch)
            print(f"Получены feature maps для слоёв: {list(feature_maps.keys())}")
        except Exception as e:
            print("Ошибка загрузки/инференса:", e)

def on_controls_change(_):
    with output:
        clear_output(wait=True)
        try:
            layer = layer_dropdown.value
            ch = channel_slider.value
            if auto_scale.value:
                visualize_feature_map(layer, ch, None, None)
            else:
                visualize_feature_map(layer, ch, vmin_box.value if vmin_box.value is not None else None,
                                      vmax_box.value if vmax_box.value is not None else None)
        except Exception as e:
            print("Сначала загрузите изображение и нажмите 'Загрузить и прогнать'. Ошибка:", e)

load_btn.on_click(on_load_clicked)
for w in [layer_dropdown, channel_slider, auto_scale, vmin_box, vmax_box]:
    w.observe(on_controls_change, names='value')

display(widgets.VBox([
    widgets.HBox([url_text]),
    widgets.HBox([upload, load_btn]),
    widgets.HBox([layer_dropdown, channel_slider]),
    widgets.HBox([auto_scale, vmin_box, vmax_box]),
    output
]))


Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]



**Подсказка:** попробуй переключать слои и каналы. Сравни, как меняются паттерны.  
— На самых ранних слоях — полосы/кромки.  
— Дальше — углы/текстуры/части объектов.



# 🧩 NLP Demo — «Почему слова имеют смысл» (Sentence Embeddings + t‑SNE)

Минимальный интерактив:
1. Берём компактную модель эмбеддингов фраз/слов (`sentence-transformers`).
2. Вводим свой список слов/фраз.
3. Сжимаем до 2D через t‑SNE и смотрим **кластеризацию на графике**.
4. Добавляем свои примеры и наблюдаем, как они «переезжают» на карте.


In [ ]:
# ⬇️ Зависимости
!pip -q install sentence-transformers scikit-learn matplotlib ipywidgets

In [ ]:

# ▶️ Импорты и базовая логика
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Лёгкая и быстрая модель эмбеддингов
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

default_items = [
    "king", "queen", "man", "woman",
    "paris", "london", "berlin",
    "apple", "banana", "fruit",
    "cat", "dog", "animal",
    "math", "biology", "science"
]

items_text = widgets.Textarea(
    value="\n".join(default_items),
    description='Список слов/фраз:',
    layout=widgets.Layout(width='60%', height='200px')
)
perplexity_slider = widgets.IntSlider(value=5, min=5, max=50, step=1, description='Perplexity')
iters_slider = widgets.IntSlider(value=1000, min=250, max=2000, step=250, description='Iterations')
go_btn = widgets.Button(description='Вычислить и нарисовать', button_style='primary')
output = widgets.Output()

def draw_tsne(labels, embeddings, perplexity=5, n_iter=1000):
    tsne = TSNE(n_components=2, perplexity=perplexity, n_iter=n_iter, init='random', learning_rate='auto')
    xy = tsne.fit_transform(embeddings)
    xs, ys = xy[:,0], xy[:,1]
    plt.figure(figsize=(6,6))
    plt.scatter(xs, ys)
    for x, y, label in zip(xs, ys, labels):
        plt.text(x+0.5, y+0.5, label, fontsize=9)
    plt.title("t-SNE of sentence embeddings")
    plt.axis('off')
    plt.show()

def on_click(_):
    with output:
        clear_output(wait=True)
        labels = [s.strip() for s in items_text.value.split("\n") if s.strip()]
        if len(labels) < 3:
            print("Добавьте хотя бы 3 слова/фразы.")
            return
        emb = model.encode(labels, normalize_embeddings=True)
        draw_tsne(labels, emb, perplexity_slider.value, iters_slider.value)

go_btn.on_click(on_click)
display(widgets.VBox([items_text, widgets.HBox([perplexity_slider, iters_slider, go_btn]), output]))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Привет мир!

In [ ]:
print("Hello world!")

Hello world!


# 🎭 Prompting Demo — «Как параметры меняют стиль текста» (distilgpt2)

Минимальный интерактив:
1. Лёгкая генеративная модель (`distilgpt2`).
2. Меняем `temperature`, `top_k`, `top_p`, `max_new_tokens`.
3. Смотрим, как меняется **креативность и связность**.

In [ ]:
# ▶️ Импорты и пайплайн
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import ipywidgets as widgets
from IPython.display import display, clear_output

device = 0 if torch.cuda.is_available() else -1
# model_name = "distilgpt2"
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(model_name)
mdl = AutoModelForCausalLM.from_pretrained(model_name)
if torch.cuda.is_available():
    mdl = mdl.to("cuda")
gen = pipeline("text-generation", model=mdl, tokenizer=tok, device=device)

prompt_box = widgets.Textarea(
    value="2+2 is",
    description='Prompt:',
    layout=widgets.Layout(width='80%', height='120px')
)
temp_slider = widgets.FloatSlider(value=0.7, min=0.1, max=1.5, step=0.1, description='temperature')
topk_slider = widgets.IntSlider(value=50, min=0, max=200, step=5, description='top_k')
topp_slider = widgets.FloatSlider(value=0.95, min=0.1, max=1.0, step=0.05, description='top_p')
max_new_slider = widgets.IntSlider(value=80, min=10, max=300, step=10, description='max_new_tokens')
seed_box = widgets.IntText(value=42, description='seed')
go_btn = widgets.Button(description='Generate', button_style='primary')

out = widgets.Output()

def on_click(_):
    with out:
        clear_output(wait=True)
        torch.manual_seed(seed_box.value)
        res = gen(
            prompt_box.value,
            do_sample=True,
            temperature=float(temp_slider.value),
            top_k=int(topk_slider.value) if topk_slider.value>0 else None,
            top_p=float(topp_slider.value),
            max_new_tokens=int(max_new_slider.value),
            pad_token_id=tok.eos_token_id
        )[0]['generated_text']
        print(res)

display(widgets.VBox([
    prompt_box,
    widgets.HBox([temp_slider, topk_slider, topp_slider]),
    widgets.HBox([max_new_slider, seed_box, go_btn]),
    out
]))

go_btn.on_click(on_click)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

# 🎨 Diffusion Demo — «Как изображение рождается из шума» (DDPM CIFAR10‑32)

**Без токенов и больших весов**: используем компактную модель `google/ddpm-cifar10-32` из `diffusers`.
Интерактивно показываем **промежуточные шаги денойзинга** и понимаем механику диффузии на практике.

In [ ]:

# ⬇️ Зависимости
!pip -q install diffusers torch torchvision pillow ipywidgets matplotlib


In [ ]:
# ▶️ Импорты и пайплайн
import torch
from diffusers import DDPMPipeline
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)

# ▶️ UI элементы
steps_slider = widgets.IntSlider(value=25, min=5, max=100, step=5, description='Steps')
frames_slider = widgets.IntSlider(value=10, min=1, max=30, step=1, description='Frames')
seed_box = widgets.IntText(value=0, description='Seed')
go_btn = widgets.Button(description='Generate with trace', button_style='primary')
out = widgets.Output()

# ▶️ Генерация "псевдо-trace"
def denoise_with_trace(num_inference_steps=25, seed=0, num_frames=10):
    frames = []

    steps = np.linspace(
        1,
        num_inference_steps,
        min(num_frames, num_inference_steps),
        dtype=int
    )

    for t in steps:
        g = torch.Generator(device=device).manual_seed(int(seed))
        img = pipe(
            num_inference_steps=int(t),
            generator=g,
            output_type="numpy",
            return_dict=False
        )[0]
        frames.append(img[0])  # (H, W, C)

    return frames

# ▶️ Отрисовка
def show_frames(frames):
    n = len(frames)
    cols = min(5, n)
    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(2 * cols, 2 * rows))
    for i, f in enumerate(frames, 1):
        plt.subplot(rows, cols, i)
        plt.imshow((f * 255).astype(np.uint8))
        plt.axis('off')
        plt.title(f"frame {i}")

    plt.tight_layout()
    plt.show()

# ▶️ Обработчик кнопки
def on_click(_):
    with out:
        clear_output(wait=True)

        frames = denoise_with_trace(
            steps_slider.value,
            seed_box.value,
            frames_slider.value
        )

        show_frames(frames)

# ▶️ Привязка
go_btn.on_click(on_click)

# ▶️ UI
display(widgets.VBox([
    widgets.HBox([steps_slider, frames_slider, seed_box, go_btn]),
    out
]))

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
